# Aviation disruption detector

This notebook provides a simple prototype for an aviation disruption detector using public Aviation Safety Reporting System (ASRS) databases. 

## 1) Installation and setup

### 1.1) Python virtual environment

Firsly, I would recomment setting up a python virtual environment where python packages will be installed using the following command:
```bash
python3 -m venv venv
```

The current notebook would need to be started using a python kernel poiting to this virtual environment.

You can activate the virtual environment in your terminal by executing the following command:
```bash
source venv/bin/activate
```

### 1.2) Installation of python packages

Once the virtual environment is ready, you can install the necessary python libraries defined inside [requirements.txt](../requirements.txt) by executing the following notebook cell, or executing the following command  defined your terminal:

In [1]:
!pip install -r ../requirements.txt

### 1.3) Loading of python libraries
Congrats! The packages are now installed and we can proceed to import the necessary libraries by executing the following cell:

In [2]:
import pandas as pd
import numpy as np
from typing import Union
from scipy.stats import chi2

#I want pandas to display all columns for analysis later
pd.set_option("display.max_columns", None)

If the previous cell executed without problems, you can now move to the next part of the aviation dirsuption detector!


## 2) Dataset loading and quick exploration
For this simple prototype, the data have been **restricted to the period 2024-2025** since ASRS database allows to download .csv files only with a maximum of 5000 entries.
The data have been merged and preprocessed using 


In [3]:
df = pd.read_csv('../data/cleaned/260620_2109.csv')
# Restrain to year 2025 for simplicity
df = df[(df.year<=2025) & (df.year > 2023)]
df

/var/folders/1b/zmjs29qd2vzdg1nl2chvp4mh0000gn/T/ipykernel_37364/1417754016.py:1: DtypeWarning: Columns (29,32,46,69,70,75,76,81,82,83,86,108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/cleaned/260620_2109.csv')


,acn,time_date,place_state_reference,place_relative_position_angle_radial,place_relative_position_distance_nautical_miles,place_altitude_agl_single_value,place_altitude_msl_single_value,place_latitude_longitude_uas,environment_flight_conditions,environment_weather_elements_visibility,environment_work_environment_factor,environment_light,environment_ceiling,environment_rvr_single_value,aircraft1_atc_advisory,aircraft1_aircraft_operator,aircraft1_make_model_name,aircraft1_aircraft_zone,aircraft1_crew_size,aircraft1_operating_under_far_part,aircraft1_flight_plan,aircraft1_mission,aircraft1_nav_in_use,aircraft1_flight_phase,aircraft1_route_in_use,aircraft1_airspace,aircraft1_maintenance_status_maintenance_deferred,aircraft1_maintenance_status_records_complete,aircraft1_maintenance_status_released_for_service,aircraft1_maintenance_status_required_correct_doc_on_board,aircraft1_maintenance_status_maintenance_type,aircraft1_maintenance_status_maintenance_items_involved,aircraft1_cabin_lighting,aircraft1_number_of_seats_number,aircraft1_passengers_on_board_number,aircraft1_crew_size_flight_attendant_number_of_crew,aircraft1_airspace_authorization_provider_uas,aircraft1_operating_under_waivers_exemptions_authorizations_uas,aircraft1_waivers_exemptions_authorizations_uas,aircraft1_airworthiness_certification_uas,aircraft1_weight_category_uas,aircraft1_configuration_uas,aircraft1_flight_operated_as_uas,aircraft1_flight_operated_with_visual_observer_uas,aircraft1_control_mode_uas,aircraft1_flying_in_near_over_uas,aircraft1_passenger_capable_uas,aircraft1_type_uas,aircraft1_number_of_uas_being_controlled_uas,component_aircraft_component,component_manufacturer,component_aircraft_reference,component_problem,aircraft2_atc_advisory,aircraft2_aircraft_operator,aircraft2_make_model_name,aircraft2_aircraft_zone,aircraft2_crew_size,aircraft2_operating_under_far_part,aircraft2_flight_plan,aircraft2_mission,aircraft2_nav_in_use,aircraft2_flight_phase,aircraft2_route_in_use,aircraft2_airspace,aircraft2_maintenance_status_maintenance_deferred,aircraft2_maintenance_status_records_complete,aircraft2_maintenance_status_released_for_service,aircraft2_maintenance_status_required_correct_doc_on_board,aircraft2_maintenance_status_maintenance_type,aircraft2_maintenance_status_maintenance_items_involved,aircraft2_cabin_lighting,aircraft2_number_of_seats_number,aircraft2_passengers_on_board_number,aircraft2_crew_size_flight_attendant_number_of_crew,aircraft2_airspace_authorization_provider_uas,aircraft2_operating_under_waivers_exemptions_authorizations_uas,aircraft2_waivers_exemptions_authorizations_uas,aircraft2_airworthiness_certification_uas,aircraft2_weight_category_uas,aircraft2_configuration_uas,aircraft2_flight_operated_as_uas,aircraft2_flight_operated_with_visual_observer_uas,aircraft2_control_mode_uas,aircraft2_flying_in_near_over_uas,aircraft2_passenger_capable_uas,aircraft2_type_uas,aircraft2_number_of_uas_being_controlled_uas,person1_location_of_person,person1_location_in_aircraft,person1_reporter_organization,person1_function,person1_qualification,person1_experience,person1_cabin_activity,person1_human_factors,person1_communication_breakdown,person1_uas_communication_breakdown,person1_asrs_report_number_accession_number,person2_location_of_person,person2_location_in_aircraft,person2_reporter_organization,person2_function,person2_qualification,person2_experience,person2_cabin_activity,person2_human_factors,person2_communication_breakdown,person2_uas_communication_breakdown,person2_asrs_report_number_accession_number,events_anomaly,events_miss_distance,events_were_passengers_involved_in_event,events_detector,events_when_detected,events_result,assessments_contributing_factors_situations,assessments_primary_problem,report1_narrative,report1_callback,report2_narrative,report2_callback,report1_synopsis,place_locale_reference_name,place_locale_reference_type,report_month,year,month,time_of_day_bucket
0,2198935,202412,US,NaN,NaN,NaN,5000.0,NaN,VMC,NaN,NaN,

## 3) Detection of possible disruptions in aviation data

To effectively detect possible disruptions in aviation data, one can consider to **use past data to model upcoming data** for a certain amount of time.

### 3.1) Problem framing and dataset preparation

We can consider our **test** sample as the last 3 months of 2025, while the modeling data as the priod 2024.01 - 2025.09

In [4]:
#Number of months to test
nmonths = 3

#Define selection for test dataframe
sel_test=(df.month > (12-nmonths)) & (df.year ==2025) #Select only the last 3 months of 2025 ()
#Define selection for model dataframe (i.e. opposite of sel_test)
sel_model=~sel_test

#Define dataframe
df_test = df[sel_test]
df_model = df[sel_model]

#Diagnostic check of df_test and df_model content
print('Dataframe counts in each year and month (test):')
print(df_test.groupby(['year','month']).size().reset_index(name='count').sort_values(['year','month']))
print('Dataframe counts in each year and month (model):')
print(df_model.groupby(['year','month']).size().reset_index(name='count').sort_values(['year','month']))

Dataframe counts in each year and month (test):
   year  month  count
0  2025     10    541
1  2025     11    426
2  2025     12    470
Dataframe counts in each year and month (model):
    year  month  count
0   2024      1    412
1   2024      2    377
2   2024      3    500
3   2024      4    500
4   2024      5    522
5   2024      6    503
6   2024      7    470
7   2024      8    493
8   2024      9    389
9   2024     10    445
10  2024     11    392
11  2024     12    424
12  2025      1    437
13  2025      2    485
14  2025      3    544
15  2025      4    562
16  2025      5    511
17  2025      6    540
18  2025      7    505
19  2025      8    551
20  2025      9    537


If everything is good we should see in our `df_test` the counts relative to Oct-Dec 2025, and Jan.2024-Sep.2025 for `df_model`.

### 3.2) Creating predictions for test dataframe

Now that we have split our data, it's time to create predictions allowing us to test the presence of statistical anomalies inside `df_test`.

I will use a very simple and common method in particle physics, called a **data-driven ABCD method**. The idea is that, for a certain portion of data called "D" and defined by the previous time selection plus another selection $S$, we can make a prediction of expected data in D using the data inside selections A,B and C (see also figure). 

![acbd](../figures/abcd.png)

In [83]:
def generate_counts_per_selection(
    df_model : pd.DataFrame,
    df_test : pd.DataFrame, 
    variable : Union[list,str]):
    counts_test = (
        df_test
        .groupby(variable)
        .size()
        .rename("count")
    )

    counts_model = (
        df_model
        .groupby(variable)
        .size()
        .rename("count")
    )

    # Ensuring that index is the same inside both df_model and df_test
    all_places = counts_model.index.union(counts_test.index)

    counts_test = (
        counts_test
        .reindex(all_places, fill_value=0) #Fill empty indices with zeros if missing
        .reset_index()
        .sort_values(variable)
    )

    counts_model = (
        counts_model
        .reindex(all_places, fill_value=0) #Fill empty indices with zeros if missing
        .reset_index()
        .sort_values(variable)
    )

    return (counts_model, counts_test)

def generate_counts_and_predictions(
    df_model : pd.DataFrame,
    df_test : pd.DataFrame, 
    variable : Union[list,str]):
    counts_model, counts_test = generate_counts_per_selection(df_model, df_test, variable)

    a_over_c = counts_test['count'].sum()/counts_model['count'].sum()
    pred_d = counts_model['count'].to_numpy('float') * a_over_c
    #Make sure that the minimum is 0.01 rather than 0
    pred_d[pred_d==0.] = 0.1
    counts_test['count_pred'] = pred_d 

    return counts_test


Let's verify that this works by testing a prediction for the various airports in the dataset and stored in the variable `place_locale_reference_name`.

In [84]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable='place_locale_reference_name')
counts_test.sort_values('count', ascending=False)

,place_locale_reference_name,count,count_pred
902,ZZZ,808,832.245002
905,ZZZZ,43,52.820722
602,ORD,31,13.452999
723,SFO,14,9.346294
271,EWR,14,7.788578
...,...,...,...
354,HHR,0,0.141611
355,HIB,0,0.141611
356,HII,0,0.283221
357,HIKES,0,0.141611


Great! The prediction seems to work since many counts seems to match the predictions. The question now becomes: how can I confidently know if these are just statistical fluctuations or real anomalies?

The answer is simple! I must perform a statistical test!

## 4) Statistical tests

It's time now to verify whether the counts and predictions are in agreement within a certain confidence level. For this I will define a test statistic based on a counting experiement described by a Poissonian distribution.

### Pearson $\chi^2$ test
For a Pearson $\chi^2$ test, the for 1 bin the statistic is defined by the $\chi^2_\text{Pearson}$ distribution:
$$
\chi^2_\text{Pearson} = \frac{(O_i - E_i)^2}{E_i}
$$
where $O_i$ is the observed counts in bin $i$ while $E_i$ is the expected counts in the same bin (i.e. the prediciton extracted with the ABCD method).

The statistical significance can be approximated with
$$
Z_\text{Pearson} = \sqrt{\chi^2_\text{Pearson}}
$$

### Poisson $\chi^2$ test
The pearson test is valid only when the prediction counts are $\ge 5$. For counts closer to zero, it's better to have a more precise estimation based on the Poissonian likelihood
$$
L(\mu | n) = \frac{\mu^n e^{-\mu}}{n!}.
$$
It's easy to show that, for an observed count of $n$ the maximum likelihood estimation is $\hat{\mu} = n$.

Therefore, we can build the poissonian test statistic $q$ as
$$
q = -2 \ln (\lambda(\mu))
$$
where $\lambda(\mu)$ is the likelihood ratio:

$$
\lambda(\mu) = \frac{L(\mu)}{L(\hat{\mu})} = \left(\frac{\mu}{n}\right)^n e^{n-\mu}
$$
and then
$$
q = \chi^2_\text{Poiss} = 2 \left[\mu -n + n \ln\left(\frac{n}{\mu}\right)\right].
$$

The Poissonian statistical significance can be approximated as:
$$
Z_\text{Poiss} = \sqrt{q}
$$

We can now define a new dataframe including the statistical significances, and sort them according to the highest significance:

In [85]:
def generate_statistical_tests(df_counts):

    df_counts = df_counts.copy()

    numerator = (df_counts['count'] - df_counts['count_pred'])
    denominator = df_counts['count_pred']
    chi_2 = np.divide(
        numerator**2,
        denominator,
        out=np.zeros_like(numerator, dtype=float),
        where=denominator != 0
    )
    sig = np.divide(
        numerator,
        np.sqrt(denominator),
        out=np.zeros_like(numerator, dtype=float),
        where=denominator != 0
    )

    def chi2_poisson(n,mu):
        x = np.divide(n,mu, out=np.zeros_like(mu, dtype=float), where=(mu!=0))
        x = np.log(x, out=np.zeros_like(n, dtype=float), where=(n!=0))
        x = n * x
        x = mu - n +x
        x*=2*np.sign(n - mu)
        return x

    df_counts['sig_pear'] = sig
    df_counts['chi2_pear'] = chi_2
    df_counts["chi2_pear_cdf"] = chi2.cdf(df_counts["chi2_pear"], df=1)
    df_counts["chi2_pear_pvalue"] = chi2.sf(df_counts["chi2_pear"], df=1)
    df_counts['chi2_poiss'] = chi2_poisson(df_counts['count'], df_counts['count_pred'])
    df_counts['sig_poiss'] = np.sqrt(df_counts['chi2_poiss'])
    df_counts["chi2_poiss_cdf"] = chi2.cdf(df_counts["chi2_poiss"], df=1)
    df_counts["chi2_poiss_pvalue"] = chi2.sf(df_counts["chi2_poiss"], df=1)

    return df_counts



## 5) Results

We can apply this method using some interesting variables stored monodimensional selection of the dataset. Let's start with the various airports stored inside `place_locale_reference_name`.

### 5.1) Testing various airports

As done before, as a first variable let's see if there is anything anomalous the various airports stored in the variable `place_locale_reference_name` for the period Oct-Dec 2026.

In [87]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable='place_locale_reference_name')
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,place_locale_reference_name,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
801,TTZP,10,0.141611,26.197402,686.303857,1.000000,2.844869e-151,65.428420,8.088784,1.000000,6.026355e-16
602,ORD,31,13.452999,4.784024,22.886885,0.999998,1.718202e-06,16.662677,4.081994,0.999955,4.465090e-05
350,HEF,2,0.100000,6.008328,36.100000,1.000000,1.874468e-09,8.182929,2.860582,0.995771,4.228642e-03
634,PIE,2,0.100000,6.008328,36.100000,1.000000,1.874468e-09,8.182929,2.860582,0.995771,4.228642e-03
787,TNCC,2,0.100000,6.008328,36.100000,1.000000,1.874468e-09,8.182929,2.860582,0.995771,4.228642e-03
...,...,...,...,...,...,...,...,...,...,...,...
351,HEG,0,0.141611,-0.376312,0.141611,0.293315,7.066852e-01,-0.283221,NaN,0.000000,1.000000e+00
352,HFD,0,0.141611,-0.376312,0.141611,0.293315,7.066852e-01,-0.283221,NaN,0.000000,1.000000e+00
353,HFY,0,0.141611,-0.376312,0.141611,0.293315,7.066852e-01,-0.283221,NaN,0.000000,1.000000e+00
355,HIB,0,0.141611,-0.376312,0.141611,0.293315,7.066852e-01,-0.283221,NaN,0.000000,1.000000e+00


You can see that several anomalies were found! If we assume a significance larger than $3\sigma$ in a standard gaussian (p-value 0.01), we see that the most anomalous counts are airports `TTZP` and `ORD`.

Let's check what could have possibly gone wrong inside there!

In [88]:
df_ttzp = df_test[df_test.place_locale_reference_name=='TTZP']

#Checking the report1_synopsis to have a short information of what could have gone wrond there:
df_ttzp.report1_synopsis.to_dict()

{5113: 'Air carrier Captain reported experiencing a GPS jamming event on approach to TTPP/POS airport.',
 5239: 'Air carrier pilot reported a suspected GPS jamming event during cruise. Situation returned to normal and flight continued.',
 5522: 'Air carrier Captain reported while in cruise flight in Piarco airspace they experienced apparent GPS jamming resulting in the flight operating with a single GPS for the remainder of the flight.',
 5562: 'Air carrier Captain reported a possible GPS jamming event near VOKAV Intersection in the vicinity of Venezuelan airspace.',
 5639: 'Air carrier Captain reported GPS jamming during cruise. Flight entered Class 1 navigation and flight continued normally.',
 5661: 'Air carrier Captain reported GPS jamming during descent. Flight crew referenced checklist and continued to a landing.',
 5662: 'Air carrier Captain reported GPS malfunction during the climb in international airspace. The flight crew continued the departure and the GPS returned to normal

Prety much all entries mention GPS jamming. So it seems that there were some GPS interferences there in that time period.

Let's check now `ORD`...

In [89]:
df_ord = df_test[df_test.place_locale_reference_name=='ORD']

#Checking the report1_synopsis to have a short information of what could have gone wrond there:
df_ord.report1_synopsis.to_dict()

{4661: 'Part 107 UAS pilot reported flying in airspace with UAS restrictions near a Class B airport.',
 4726: 'Air carrier Captain reported a ground conflict at ORD airport requiring the Captain to immediately stop the aircraft short of the runway for a fast approaching aircraft on Runway 28C.',
 5138: 'Air carrier ground personnel reported unsafe operation of vehicle in close proximity to aircraft and people during push back.',
 5199: 'Air carrier flight crew reported that nighttime conditions along with poor signage at ORD led to the flight crew turning onto the wrong taxiway.',
 5214: 'Ground Vehicle Driver reported their de-icing truck sustained damage due to the hazardous placement of the Jersey walls near Taxiway B4 at ORD.',
 5273: 'Air carrier Captain reported uncommanded movement by the push crew while at the gate. Flight continued the push back.',
 5299: 'Air carrier Captain reported abruptly stopping the aircraft due to ground vehicle on service road failing to yield to the 

It seems that the 'GPWS' system was mentioned many times in this reports (7 to be precise). So this could indicate a problem with that system.

In [90]:
df_ord.report1_synopsis.apply(lambda x: 'GPWS' in x).sum()

np.int64(7)

We can also verify this hypothesis by increasing the granularity of the prediction, and splitting data for `['place_locale_reference_name','component_aircraft_component']`

In [93]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable=['place_locale_reference_name','component_aircraft_component'])
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,place_locale_reference_name,component_aircraft_component,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
223,TTZP,GPS & Other Satellite Navigation,9,0.100000,28.144271,792.100000,1.000000,2.816268e-174,63.196574,7.949627,1.000000,1.870735e-15
162,ORD,GPWS,8,0.100000,24.981994,624.100000,1.000000,9.594598e-138,54.312426,7.369696,1.000000,1.710170e-13
387,ZZZ,Door Warning System,4,0.316197,6.551145,42.917505,1.000000,5.709747e-11,12.933859,3.596367,0.999677,3.226918e-04
626,ZZZ,UAS Component Unknown / Undifferentiated,7,1.264789,5.099649,26.006415,1.000000,3.402847e-07,12.483642,3.533220,0.999589,4.105311e-04
314,ZZZ,Aerofoil Ice System,4,0.474296,5.119424,26.208501,1.000000,3.064704e-07,10.006336,3.163279,0.998440,1.560026e-03
...,...,...,...,...,...,...,...,...,...,...,...,...
268,ZGSZ,Navigational Equipment and Processing,0,0.158099,-0.397616,0.158099,0.309087,6.909131e-01,-0.316197,NaN,0.000000,1.000000e+00
269,ZGZU,GPS & Other Satellite Navigation,0,0.158099,-0.397616,0.158099,0.309087,6.909131e-01,-0.316197,NaN,0.000000,1.000000e+00
270,ZHU,FMS/FMC,0,0.158099,-0.397616,0.158099,0.309087,6.909131e-01,-0.316197,NaN,0.000000,1.000000e+00
272,ZHU,Navigation Database,0,0.316197,-0.562314,0.316197,0.426098,5.739019e-01,-0.632395,NaN,0.000000,1.000000e+00


### 5.2) Let's split by problem, rather than local airport

Let's repeat now with another variable. Looking at the data there is `assessments_primary_problem`. Let's see if any specific issue became more prominent at some point

In [103]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable='assessments_primary_problem')
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,assessments_primary_problem,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
7,Environment - Non Weather Related,41,20.757951,4.442856,19.738969,0.999991,0.000009,15.328604,3.915176,0.999910,0.000090
16,Staffing,2,0.284355,3.217337,10.351254,0.998706,0.001294,4.371420,2.090794,0.963453,0.036547
2,Airport,57,43.222035,2.095716,4.392026,0.963893,0.036107,3.987965,1.996989,0.954174,0.045826
1,Aircraft,497,453.831368,2.026381,4.106219,0.957274,0.042726,3.981883,1.995466,0.954008,0.045992
3,Airspace Structure,26,18.625285,1.708810,2.920032,0.912514,0.087486,2.596547,1.611380,0.892903,0.107097
5,Chart Or Publication,24,17.630041,1.517085,2.301548,0.870755,0.129245,2.065659,1.437240,0.849350,0.150650
17,Weather,69,59.003765,1.301357,1.693531,0.806864,0.193136,1.605254,1.266986,0.794840,0.205160
0,ATC Equipment / Nav Facility / Buildings,26,22.748440,0.681736,0.464764,0.504594,0.495406,0.444078,0.666392,0.494839,0.505161
13,Manuals,3,1.990488,0.715536,0.511992,0.525722,0.474278,0.442370,0.665109,0.494019,0.505981
4,Ambiguous,154,148.860101,0.421275,0.177472,0.326446,0.673554,0.175464,0.418885,0.324700,0.675300


Here I see a clear excess for `Environment - Non Weather Related`. Let's investigate this:

In [96]:
df_env = df_test[df_test.assessments_primary_problem=='Environment - Non Weather Related']
df_env.report1_synopsis.to_dict()
df_env

,acn,time_date,place_state_reference,place_relative_position_angle_radial,place_relative_position_distance_nautical_miles,place_altitude_agl_single_value,place_altitude_msl_single_value,place_latitude_longitude_uas,environment_flight_conditions,environment_weather_elements_visibility,environment_work_environment_factor,environment_light,environment_ceiling,environment_rvr_single_value,aircraft1_atc_advisory,aircraft1_aircraft_operator,aircraft1_make_model_name,aircraft1_aircraft_zone,aircraft1_crew_size,aircraft1_operating_under_far_part,aircraft1_flight_plan,aircraft1_mission,aircraft1_nav_in_use,aircraft1_flight_phase,aircraft1_route_in_use,aircraft1_airspace,aircraft1_maintenance_status_maintenance_deferred,aircraft1_maintenance_status_records_complete,aircraft1_maintenance_status_released_for_service,aircraft1_maintenance_status_required_correct_doc_on_board,aircraft1_maintenance_status_maintenance_type,aircraft1_maintenance_status_maintenance_items_involved,aircraft1_cabin_lighting,aircraft1_number_of_seats_number,aircraft1_passengers_on_board_number,aircraft1_crew_size_flight_attendant_number_of_crew,aircraft1_airspace_authorization_provider_uas,aircraft1_operating_under_waivers_exemptions_authorizations_uas,aircraft1_waivers_exemptions_authorizations_uas,aircraft1_airworthiness_certification_uas,aircraft1_weight_category_uas,aircraft1_configuration_uas,aircraft1_flight_operated_as_uas,aircraft1_flight_operated_with_visual_observer_uas,aircraft1_control_mode_uas,aircraft1_flying_in_near_over_uas,aircraft1_passenger_capable_uas,aircraft1_type_uas,aircraft1_number_of_uas_being_controlled_uas,component_aircraft_component,component_manufacturer,component_aircraft_reference,component_problem,aircraft2_atc_advisory,aircraft2_aircraft_operator,aircraft2_make_model_name,aircraft2_aircraft_zone,aircraft2_crew_size,aircraft2_operating_under_far_part,aircraft2_flight_plan,aircraft2_mission,aircraft2_nav_in_use,aircraft2_flight_phase,aircraft2_route_in_use,aircraft2_airspace,aircraft2_maintenance_status_maintenance_deferred,aircraft2_maintenance_status_records_complete,aircraft2_maintenance_status_released_for_service,aircraft2_maintenance_status_required_correct_doc_on_board,aircraft2_maintenance_status_maintenance_type,aircraft2_maintenance_status_maintenance_items_involved,aircraft2_cabin_lighting,aircraft2_number_of_seats_number,aircraft2_passengers_on_board_number,aircraft2_crew_size_flight_attendant_number_of_crew,aircraft2_airspace_authorization_provider_uas,aircraft2_operating_under_waivers_exemptions_authorizations_uas,aircraft2_waivers_exemptions_authorizations_uas,aircraft2_airworthiness_certification_uas,aircraft2_weight_category_uas,aircraft2_configuration_uas,aircraft2_flight_operated_as_uas,aircraft2_flight_operated_with_visual_observer_uas,aircraft2_control_mode_uas,aircraft2_flying_in_near_over_uas,aircraft2_passenger_capable_uas,aircraft2_type_uas,aircraft2_number_of_uas_being_controlled_uas,person1_location_of_person,person1_location_in_aircraft,person1_reporter_organization,person1_function,person1_qualification,person1_experience,person1_cabin_activity,person1_human_factors,person1_communication_breakdown,person1_uas_communication_breakdown,person1_asrs_report_number_accession_number,person2_location_of_person,person2_location_in_aircraft,person2_reporter_organization,person2_function,person2_qualification,person2_experience,person2_cabin_activity,person2_human_factors,person2_communication_breakdown,person2_uas_communication_breakdown,person2_asrs_report_number_accession_number,events_anomaly,events_miss_distance,events_were_passengers_involved_in_event,events_detector,events_when_detected,events_result,assessments_contributing_factors_situations,assessments_primary_problem,report1_narrative,report1_callback,report2_narrative,report2_callback,report1_synopsis,place_locale_reference_name,place_locale_reference_type,report_month,year,month,time_of_day_bucket
4655,2291202,202510,US,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,

I see that there are quite some entreis filled for `aircraft2_make_model_name`. Let's move the problem to 2 dimensions!

In [97]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable=['assessments_primary_problem','aircraft2_make_model_name'])
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,assessments_primary_problem,aircraft2_make_model_name,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
199,Environment - Non Weather Related,UAV: Unpiloted Aerial Vehicle,13,0.745418,14.193801,201.463997,1.000000,1.000831e-45,49.818599,7.058229,1.000000,1.686378e-12
65,Airspace Structure,Helicopter,5,0.496945,6.387826,40.804322,1.000000,1.682605e-10,14.081028,3.752470,0.999825,1.751011e-04
12,Aircraft,Cirrus Aircraft Undifferentiated,2,0.100000,6.008328,36.100000,1.000000,1.874468e-09,8.182929,2.860582,0.995771,4.228642e-03
115,Ambiguous,B787 Dreamliner Undifferentiated or Other Model,2,0.124236,5.321744,28.320958,1.000000,1.027772e-07,7.363342,2.713548,0.993343,6.656694e-03
52,Airspace Structure,Amateur/Home Built/Experimental,2,0.124236,5.321744,28.320958,1.000000,1.027772e-07,7.363342,2.713548,0.993343,6.656694e-03
...,...,...,...,...,...,...,...,...,...,...,...,...
181,Company Policy,Small Aircraft; High Wing; 1 Eng; Fixed Gear,0,0.124236,-0.352472,0.124236,0.275515,7.244846e-01,-0.248473,NaN,0.000000,1.000000e+00
180,Company Policy,Commercial Fixed Wing,0,0.248473,-0.498470,0.248473,0.381847,6.181527e-01,-0.496945,NaN,0.000000,1.000000e+00
179,Company Policy,Cessna Stationair/Turbo Stationair 7/8,0,0.124236,-0.352472,0.124236,0.275515,7.244846e-01,-0.248473,NaN,0.000000,1.000000e+00
177,Company Policy,Airbus Industrie Undifferentiated or Other Model,0,0.124236,-0.352472,0.124236,0.275515,7.244846e-01,-0.248473,NaN,0.000000,1.000000e+00


It seems that indeed that a lot of these entries are due to drones `aircraft2_make_model_name='UAV: Unpiloted Aerial Vehicle'`. Printing now the report1_synopsis:

In [98]:
df_test[(df_test.assessments_primary_problem=='Environment - Non Weather Related') & (df_test.aircraft2_make_model_name.apply(lambda x: 'UAV' in x if isinstance(x,str) else False))].report1_synopsis.to_dict()

{4708: 'Air taxi pilot reported an NMAC with a UAS during cruise flight. No evasive action was taken.',
 4729: 'Air carrier First Officer reported an NMAC during arrival with what was possibly a UAS. No evasive action was taken.',
 4751: 'Flight Instructor reported an NMAC with a UAS during initial approach. No evasive action was taken.',
 4908: 'Air carrier Captain reported an NMAC with a UAS during arrival descent. No evasive action was taken.',
 4917: 'Air carrier Captain reported the First Officer noticed a UAS during arrival. No evasive action was taken.',
 4931: 'General aviation pilot reported an NMAC with a UAS during descent. The pilot took evasive action to avoid collision.',
 4997: 'Air carrier flight crew reported an NMAC with a UAS during climb. No evasive action was taken.',
 5142: 'Air carrier Captain reported the First Officer called out an NMAC with a UAS during initial approach. No evasive action was taken.',
 5160: 'Air carrier Captain reported an NMAC with a UAS dur

A lot of possible "near mid-air collision" (NMAC) due to drones. This reports a trend that drone usage might be more and more dangerous for air carriers.

### 5.3) Environment light


In [99]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable='environment_work_environment_factor')
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,environment_work_environment_factor,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
5,Poor Lighting,10,5.368421,1.998968,3.995872,0.954388,0.045612,3.177867,1.782657,0.925358,0.074642
0,Excessive Humidity,1,0.100000,2.846050,8.100000,0.995573,0.004427,2.805170,1.674864,0.906039,0.093961
2,Excessive Wind (UAS),3,2.368421,0.410391,0.168421,0.318481,0.681519,0.155175,0.393922,0.306362,0.693638
1,Excessive Humidity; Temperature - Extreme,0,0.157895,-0.397360,0.157895,0.308898,0.691102,-0.315789,NaN,0.000000,1.000000
3,Glare,2,3.473684,-0.790695,0.625199,0.570878,0.429122,-0.739094,NaN,0.000000,1.000000
4,Glare; Poor Lighting,0,0.157895,-0.397360,0.157895,0.308898,0.691102,-0.315789,NaN,0.000000,1.000000
6,Poor Lighting; Glare,0,0.157895,-0.397360,0.157895,0.308898,0.691102,-0.315789,NaN,0.000000,1.000000
7,Temperature - Extreme,2,6.315789,-1.717301,2.949123,0.914076,0.085924,-4.031957,NaN,0.000000,1.000000


It doesnt' seem that there is any special trend in the environment work factors... Let's test another varible

In [100]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable='environment_light')
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,environment_light,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
3,Night,100,70.338508,3.536685,12.508143,0.999595,0.000405,11.047170,3.323728,0.999112,0.000888
0,Dawn,14,6.405608,3.000632,9.003795,0.997306,0.002694,6.703952,2.589199,0.990380,0.009620
1,Daylight,364,400.473711,-1.822607,3.321895,0.931637,0.068363,-3.427603,NaN,0.000000,1.000000
2,Dusk,14,14.782173,-0.203439,0.041387,0.161208,0.838792,-0.042137,NaN,0.000000,1.000000


Interestingly, it seems that we have many more night counts in this 3 months period. Does this mean that for some reason airplanes got more subject to issues at night?

Probably not since these are the winter months and, since the dataset seems to be dominated by US data, The reason why nights have a higher rate of events it's probably just due to the fact that nights are longer in winter. :-)

I verified this hypothesis by building the prediction from winter months of 2024, rather than from Jan-2024-Sep 2025 to see if the excess of Nights disappeared.

In [101]:
#Number of months to test
nmonths = 3

#Define selection for test dataframe
sel_test_2=(df.month > (12-nmonths)) & (df.year ==2025) #Select only the last 3 months of 2025 ()
#Define selection for model dataframe (i.e. opposite of sel_test)
sel_model_2=(df.month > (12-nmonths)) & (df.year ==2024)

#Define dataframe
df_test_2 = df[sel_test_2]
df_model_2 = df[sel_model_2]

In [102]:
counts_test_2 = generate_counts_and_predictions(df_model_2, df_test_2, variable='environment_light')
counts_test_with_stats = generate_statistical_tests(counts_test_2)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,environment_light,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
0,Dawn,14,11.245714,0.821326,0.674576,0.588539,0.411461,0.625395,0.790819,0.570950,0.429050
3,Night,100,98.400000,0.161296,0.026016,0.128139,0.871861,0.025876,0.160861,0.127797,0.872203
1,Daylight,364,362.674286,0.069613,0.004846,0.055498,0.944502,0.004840,0.069571,0.055465,0.944535
2,Dusk,14,19.680000,-1.280371,1.639350,0.799585,0.200415,-1.824724,NaN,0.000000,1.000000


You can see that now the significance is well below 1 for all entries. :-) So, the system has detected an effective trend, but not something which is useful for aviation.

### 5.4) Other

In [63]:
counts_test = generate_counts_and_predictions(df_model, df_test, variable=['aircraft1_make_model_name','aircraft1_flight_phase','person1_location_in_aircraft'])
counts_test_with_stats = generate_statistical_tests(counts_test)
counts_test_with_stats.sort_values('chi2_poiss_pvalue')

/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/valentem/Nextcloud/Documents/Applications/2026-05-26/IATA/Interviews/Round 2/disruptive_signals_detector/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,aircraft1_make_model_name,aircraft1_flight_phase,person1_location_in_aircraft,count,count_pred,sig_pear,chi2_pear,chi2_pear_cdf,chi2_pear_pvalue,chi2_poiss,sig_poiss,chi2_poiss_cdf,chi2_poiss_pvalue
833,Commercial Fixed Wing,Parked,Door Area,27,6.179762,8.375289,70.145469,1.000000,5.508842e-17,37.985607,6.163247,1.000000,7.126843e-10
867,DA40 Diamond Star,Climb,Flight Deck,9,1.293439,6.776224,45.917209,1.000000,1.233575e-11,19.505443,4.416497,0.999990,1.003134e-05
286,B737-800,Parked,Door Area,5,0.718577,5.050699,25.509561,1.000000,4.401962e-07,10.836357,3.291862,0.999005,9.952633e-04
846,Commercial Fixed Wing,Taxi,Flight Deck,56,35.066557,3.535038,12.496495,0.999592,4.077162e-04,10.560741,3.249729,0.998845,1.155149e-03
808,Commercial Fixed Wing,Final Approach,Flight Deck,38,21.988456,3.414569,11.659280,0.999361,6.388307e-04,9.554124,3.090975,0.998005,1.995007e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...
692,Champion Citabria 7ECA,Landing,Flight Deck,0,0.718577,-0.847689,0.718577,0.603389,3.966111e-01,-1.437154,NaN,0.000000,1.000000e+00
690,Challenger Jet Undifferentiated or Other Model,Parked,Flight Deck,0,0.143715,-0.379098,0.143715,0.295385,7.046150e-01,-0.287431,NaN,0.000000,1.000000e+00
689,Challenger CL604,Initial Climb,Flight Deck,0,0.143715,-0.379098,0.143715,0.295385,7.046150e-01,-0.287431,NaN,0.000000,1.000000e+00
704,Cirrus Aircraft Undifferentiated,Final Approach,Flight Deck,0,0.143715,-0.379098,0.143715,0.295385,7.046150e-01,-0.287431,NaN,0.000000,1.000000e+00


In [67]:
df_test[(df_test.aircraft1_make_model_name=='DA40 Diamond Star') & (df_test.aircraft1_flight_phase=='Climb')][['report1_synopsis','place_locale_reference_name']].to_dict(orient='index')

{4790: {'report1_synopsis': 'DA40 Flight Instructor and student pilot reported safely returning to the departure airport after experiencing engine roughness and oscillating RPMs during climb.',
  'place_locale_reference_name': 'ZZZ'},
 4915: {'report1_synopsis': 'DA-40 Flight Instructor reported an engine malfunction during the climb on a training flight. The instructor took the controls and returned to the departure airport; landing safely on the opposite direction runway.',
  'place_locale_reference_name': 'ZZZ'},
 5106: {'report1_synopsis': 'DA40 Flight Instructor and student reported fuel leak from wing gas cap due to over-fueling. Flight returned to departure airport to defuel and depart.',
  'place_locale_reference_name': 'ZZZ'},
 5207: {'report1_synopsis': 'DA40 pilot reported entering icing conditions without activating pitot heat. In the ensuing high workload; the airspeed indicator was lost; an altitude deviation occurred; and the pilot regained control and safely diverted to

This might indicate an issue with a specific aircraft DA40 Diamond Star. However, seems all the incidents seems to come from 'ZZZ' it's well possible that this issue is simply due to the same aircraft having multiple problems.

In [73]:
df_test[(df_test.aircraft1_make_model_name=='Commercial Fixed Wing') & (df_test.aircraft1_flight_phase=='Parked') & (df_test.person1_location_in_aircraft=='Door Area')][['report1_synopsis','place_locale_reference_name']].to_dict(orient='index')

{4639: {'report1_synopsis': 'Air carrier Flight Attendant reported forgetting to change the streamer on the door to the disarmed position after landing. Catering called the cabin to confirm the door was disarmed.',
  'place_locale_reference_name': nan},
 4682: {'report1_synopsis': 'Air carrier flight attendants reported catering personnel opened an armed door slightly; but was stopped by a flight attendant; preventing an escape slide deployment prior to takeoff. Maintenance was called upon landing.',
  'place_locale_reference_name': 'ZZZ'},
 4709: {'report1_synopsis': 'Air carrier Flight Attendant reported the girt bar popped out of the bracket when door was opened; but the escape slide did not deploy.',
  'place_locale_reference_name': nan},
 4710: {'report1_synopsis': 'Air carrier Flight Attendant reported the gate agent attempted to open an armed door without knocking first. The door was then disarmed and opened with no escape slide deployment.',
  'place_locale_reference_name': nan